# Calibration outputs

Summer4 port of `tweak_cases_calib` `notebooks/05-Outputs.ipynb`.

Loads an ArviZ `.nc` and optional `_outputs.pkl` from a local or massive run,
summarises the posterior, overlays priors, and plots count/rate panels against
calibration targets. Age bands are the **model** ages (summed for totals) —
single-year regroup from summer3 is not ported.

Use the **tb-macro-summer4 (pixi)** kernel.


In [ ]:
import sys
from pathlib import Path

import arviz as az
import numpy as np
import pandas as pd

from tb_macro.constants import (
    LOCAL_OUTPUT_PATH,
    N_OUTPUT_SAMPLES,
    OUTPUT_PATH,
    SCENARIO_PARAMS,
    YOUNG_END_AGE,
)
from tb_macro.inputs import load_who_outcomes
from tb_macro.outputs import (
    get_posterior_samples,
    load_sampled_outputs,
    rerun_model_for_outputs,
)
from tb_macro.parameters import PARAM_NAMES
from tb_macro.targets import (
    LATENT_TARGET,
    NOTIF_TARGET,
    PREV_DECLINE_TARGET,
    PULM_PREV_TARGET,
)
from tb_macro.constants import ISO3

_here = Path.cwd()
for candidate in (_here, _here / "notebooks" / "summer3", _here / "summer3"):
    if (candidate / "plots.py").is_file():
        sys.path.insert(0, str(candidate))
        break

from plots import plot_outputs_panels, plot_prior_post

SOURCE = "local"  # "local" or "remote"
RUN_ID = None  # stamp like "20260917T1241Z", or None for newest
FOLDER = None  # remote task id under OUTPUT_PATH, or None for newest


In [ ]:
def _newest_nc(directory: Path) -> Path:
    files = sorted(directory.glob("*.nc"), key=lambda p: p.stat().st_mtime)
    if not files:
        raise FileNotFoundError(f"No .nc under {directory}")
    return files[-1]


if SOURCE == "local":
    run_dir = LOCAL_OUTPUT_PATH
    nc_path = run_dir / f"{RUN_ID}.nc" if RUN_ID else _newest_nc(run_dir)
elif SOURCE == "remote":
    root = OUTPUT_PATH
    if FOLDER is not None:
        run_dir = root / FOLDER
    else:
        dirs = sorted(
            (p for p in root.iterdir() if p.is_dir() and any(p.glob("*.nc"))),
            key=lambda p: p.stat().st_mtime,
        )
        if not dirs:
            raise FileNotFoundError(f"No run dirs under {root}")
        run_dir = dirs[-1]
    nc_path = run_dir / f"{RUN_ID}.nc" if RUN_ID else _newest_nc(run_dir)
else:
    raise ValueError(SOURCE)

stamp = nc_path.stem
outputs_path = nc_path.with_name(f"{stamp}_outputs.pkl")
print(f"run_dir {run_dir}")
print(f"idata   {nc_path}")
print(f"pickle  {outputs_path}  exists={outputs_path.exists()}")

idata = az.from_netcdf(nc_path)
_, _, who_mort = load_who_outcomes(ISO3)


## Posterior summary


In [ ]:
summary = az.summary(idata)
summary = summary.rename(index=lambda k: PARAM_NAMES.get(k, k))
summary[["mean", "sd", "r_hat"]]


In [ ]:
req_vars = [v for v in idata.posterior.data_vars if v != "mixing_dist_sd"]
plot_prior_post(idata, req_vars)


## Scenario outputs


In [ ]:
if outputs_path.exists():
    outputs, sample_labels = load_sampled_outputs(outputs_path)
    print(f"loaded {outputs_path.name}")
else:
    print("pickle missing; re-running posterior samples")
    samples = get_posterior_samples(idata, N_OUTPUT_SAMPLES)
    outputs, sample_labels = rerun_model_for_outputs(idata, SCENARIO_PARAMS, samples)


def _frames_to_multi(frames: list[pd.DataFrame], labels: list[str]) -> pd.DataFrame:
    parts = []
    for label, frame in zip(labels, frames, strict=True):
        wide = frame.copy()
        wide.columns = pd.MultiIndex.from_product([[label], wide.columns])
        parts.append(wide)
    return pd.concat(parts, axis=1)


scenario = 0
age_pop = _frames_to_multi(outputs[scenario]["age_pop"], sample_labels)
incidence = _frames_to_multi(outputs[scenario]["incidence"], sample_labels)
notifications = _frames_to_multi(outputs[scenario]["notifications"], sample_labels)
prevalence = _frames_to_multi(outputs[scenario]["prevalence"], sample_labels)
deaths = _frames_to_multi(outputs[scenario]["deaths"], sample_labels)
latent = _frames_to_multi(outputs[scenario]["latent"], sample_labels)
pulm_prev = _frames_to_multi(outputs[scenario]["pulm_prev"], sample_labels)

bad = (age_pop < -1e-3).any().groupby(level=0).any()
good_labels = [lab for lab in sample_labels if not bool(bad.get(lab, False))]
print(f"samples {len(sample_labels)}  kept {len(good_labels)}")


def sum_ages(df: pd.DataFrame) -> pd.DataFrame:
    kept = df.loc[:, pd.IndexSlice[good_labels, :]]
    return kept.T.groupby(level=0).sum().T


total_pop = sum_ages(age_pop)
incs = sum_ages(incidence)
notifs = sum_ages(notifications)
prevs = sum_ages(prevalence)
tb_deaths = sum_ages(deaths)
adult_ages = [
    age
    for age in age_pop.columns.get_level_values(1).unique()
    if int(age) >= YOUNG_END_AGE
]
adult_pop = sum_ages(age_pop.loc[:, pd.IndexSlice[:, adult_ages]])
pulm = sum_ages(pulm_prev.loc[:, pd.IndexSlice[:, adult_ages]])
lat = sum_ages(latent.loc[:, pd.IndexSlice[:, adult_ages]])


In [ ]:
plot_outputs_panels(
    prevs,
    incs,
    notifs,
    NOTIF_TARGET,
    tb_deaths,
    who_mort,
    lat,
    None,
    total_pop,
    1990.0,
    2040.0,
    "count",
    pulm,
    None,
    adult_pop,
)


In [ ]:
plot_outputs_panels(
    prevs,
    incs,
    notifs,
    NOTIF_TARGET,
    tb_deaths,
    None,
    lat,
    LATENT_TARGET,
    total_pop,
    1990.0,
    2040.0,
    "rate",
    pulm,
    PULM_PREV_TARGET,
    adult_pop,
    PREV_DECLINE_TARGET,
)


## Trace


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

posterior = idata.posterior
dataset = posterior.dataset if hasattr(posterior, "dataset") else posterior
names = [v for v in dataset.data_vars if v != "mixing_dist_sd"]
n_chains = int(dataset.sizes["chain"])
fig = make_subplots(rows=len(names), cols=2, subplot_titles=[f"{n} posterior" for n in names for _ in (0, 1)][: 2 * len(names)])
for i, name in enumerate(names):
    values = np.asarray(dataset[name])
    for chain in range(n_chains):
        fig.add_trace(
            go.Scatter(y=values[chain], mode="lines", line=dict(width=1), opacity=0.7, showlegend=False),
            row=i + 1,
            col=1,
        )
        fig.add_trace(
            go.Histogram(x=values[chain], showlegend=False, opacity=0.5),
            row=i + 1,
            col=2,
        )
fig.update_layout(height=180 * len(names), width=900, template="plotly_white", title_text="Trace")
fig.show()
